## Day 11 Plan
1. Check the existing dataset structure and splits.
2. Define image resizing and normalization.
3. Prepare dataset loaders.
4. Check train/validation/test split integrity.
5. Apply useful augmentation during training.
6. Save the preprocessing configuration as Dataset V1.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
DATASET_FOLDER="/content/drive/MyDrive/Dataset (1)"
DATASETS=[
    "DIATAquarium.v4i.yolov8",
    "Aquatic Plant.v2i.yolov8",
    "well.v8i.yolov8"
]
for dataset in DATASETS:
    path=os.path.join(DATASET_FOLDER,dataset)
    print(dataset,"->",os.path.exists(path))

DIATAquarium.v4i.yolov8 -> True
Aquatic Plant.v2i.yolov8 -> True
well.v8i.yolov8 -> True


# Dataset V1 Setup

A separate Dataset V1 configuration is created while keeping the original raw datasets unchanged.

In [ ]:
import os
DATASET_V1="/content/drive/MyDrive/Dataset_V1"
os.makedirs(DATASET_V1,exist_ok=True)
print("Dataset V1 folder:",DATASET_V1)

Dataset V1 folder: /content/drive/MyDrive/Dataset_V1


##  Resizing and Normalization
Images are prepared for consistent model input. Resizing provides a common image size, while normalization keeps pixel values in a consistent range.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
IMG_SIZE=640
print("Target image size:",IMG_SIZE,"x",IMG_SIZE)
print("Pixel normalization: 0-1")

image_path=os.path.join(DATASET_FOLDER,DATASETS[0],"train","images")
file=os.listdir(image_path)[0]
img=cv2.imread(os.path.join(image_path,file))
resized=cv2.resize(img,(640,640))
normalized=resized/255.0

print("Original size:",img.shape[:2])
print("Resized size:",resized.shape[:2])
print("Normalized range:",normalized.min(),"to",normalized.max())

Target image size: 640 x 640
Pixel normalization: 0-1
Original size: (1080, 1920)
Resized size: (640, 640)
Normalized range: 0.0 to 1.0


##  Dataset Loader

A reusable dataset loader is implemented to load underwater images and their corresponding YOLO annotations.

The loader applies resizing to 640 × 640 and pixel normalization to the range 0-1 during loading.

The original raw datasets remain unchanged.

In [ ]:
import torch
from torch.utils.data import Dataset
class UnderwaterDataset(Dataset):
    def __init__(self,image_dir,label_dir,img_size=640):
        self.image_dir=image_dir
        self.label_dir=label_dir
        self.img_size=img_size
        self.files=[f for f in os.listdir(image_dir)
                    if f.lower().endswith((".jpg",".jpeg",".png"))]
    def __len__(self):
        return len(self.files)
    def __getitem__(self,idx):
        file=self.files[idx]
        image_path=os.path.join(self.image_dir,file)
        label_path=os.path.join(self.label_dir,os.path.splitext(file)[0]+".txt")
        image=cv2.imread(image_path)
        image=cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
        image=cv2.resize(image,(self.img_size,self.img_size))
        image=image.astype(np.float32)/255.0
        image=torch.from_numpy(image).permute(2,0,1)
        labels=[]
        if os.path.exists(label_path):
            with open(label_path,"r") as f:
                for line in f:
                    values=line.strip().split()
                    if len(values)==5:
                        labels.append([float(x) for x in values])
        labels=torch.tensor(labels,dtype=torch.float32).reshape(-1,5)
        return image,labels,file

In [ ]:
for dataset in DATASETS:
    image_dir=os.path.join(DATASET_FOLDER,dataset,"train","images")
    label_dir=os.path.join(DATASET_FOLDER,dataset,"train","labels")
    data=UnderwaterDataset(image_dir,label_dir,IMG_SIZE)
    image,labels,file=data[0]
    print("\nDataset:",dataset)
    print("Number of images:",len(data))
    print("Image shape:",image.shape)
    print("Label shape:",labels.shape)
    print("Pixel range:",image.min().item(),"to",image.max().item())


Dataset: DIATAquarium.v4i.yolov8
Number of images: 8121
Image shape: torch.Size([3, 640, 640])
Label shape: torch.Size([3, 5])
Pixel range: 0.0 to 1.0

Dataset: Aquatic Plant.v2i.yolov8
Number of images: 1892
Image shape: torch.Size([3, 640, 640])
Label shape: torch.Size([1, 5])
Pixel range: 0.0 to 1.0

Dataset: well.v8i.yolov8
Number of images: 3795
Image shape: torch.Size([3, 640, 640])
Label shape: torch.Size([1, 5])
Pixel range: 0.019607843831181526 to 1.0


## Train/Validation/Test Split Check

The existing train, validation and test folders were inspected. The current split structure is preserved for now. A further leakage check will be performed before Dataset V1 is finalized.

In [ ]:
for dataset in DATASETS:
    print("\nDataset:",dataset)
    for split in ["train","valid","test"]:
        path=os.path.join(DATASET_FOLDER,dataset,split,"images")
        if os.path.exists(path):
            print(split,":",len(os.listdir(path)),"images")
        else:
            print(split,": Not present")


Dataset: DIATAquarium.v4i.yolov8
train : 8121 images
valid : 773 images
test : Not present

Dataset: Aquatic Plant.v2i.yolov8
train : 1892 images
valid : 179 images
test : 89 images

Dataset: well.v8i.yolov8
train : 3795 images
valid : 191 images
test : 184 images


## Step 7 — Data Leakage Check

The purpose is to identify whether images from the same source sequence are present in different dataset splits.

If the same source appears across train, validation or test, this may cause data leakage and affect evaluation reliability.

In [ ]:
for dataset in DATASETS:
    print("\nDataset:",dataset)
    for split in ["train","valid","test"]:
        image_dir=os.path.join(DATASET_FOLDER,dataset,split,"images")
        if not os.path.exists(image_dir):
            print(split,": Not present")
            continue
        files=os.listdir(image_dir)
        print(split,":",len(files),"files")
        print("Sample filenames:")
        for f in files[:5]:
            print(" ",f)


Dataset: DIATAquarium.v4i.yolov8
train : 8121 files
Sample filenames:
  av0_0_20240222_105838_001169_jpg.rf.821418c6846b599d531751d333b3a92c.jpg
  av0_0_20240222_105838_000824_jpg.rf.40bd5ea4dca9a7e2cec4e2933c77e862.jpg
  av0_0_20240222_105838_000842_jpg.rf.086d12da29d4ca85e6bc73cd51ee1e33.jpg
  av0_0_20240222_105838_001183_jpg.rf.026b03a38a02e6f27a210dc33fbd9a63.jpg
  av0_0_20240222_105838_001174_jpg.rf.a6982010bb58affdda7f3dade2c869b1.jpg
valid : 773 files
Sample filenames:
  av0_0_20231228_094444_mp4-18_jpg.rf.34c452c8ae9deafa4fb1385a0f0f4a9c.jpg
  av0_0_20231228_094444_mp4-487_jpg.rf.17f59f7186996e06f06a9c6f7408a744.jpg
  av0_0_20231228_094444_mp4-209_jpg.rf.ce4b4bc0c129393268807ded38886e43.jpg
  av0_0_20231228_094444_mp4-383_jpg.rf.ae409989f22c651a6e1c5290d9e956d1.jpg
  av0_0_20231228_094444_mp4-404_jpg.rf.b6fb188980a510ca9aaf3500d6ea3d7f.jpg
test : Not present

Dataset: Aquatic Plant.v2i.yolov8
train : 1892 files
Sample filenames:
  2023-12-09_13_20_33_mp4-242_jpg.rf.250b6acfe99

In [ ]:
import re
def get_source_id(filename):
    name=os.path.splitext(filename)[0]
    name=re.sub(r"\.rf\.[a-f0-9]+$","",name)
    name=re.sub(r"_jpg$","",name)
    name=re.sub(r"-\d+$","",name)
    return name
for dataset in DATASETS:
    split_sources={}
    for split in ["train","valid","test"]:
        image_dir=os.path.join(DATASET_FOLDER,dataset,split,"images")
        if not os.path.exists(image_dir):
            continue
        sources=set()
        for file in os.listdir(image_dir):
            if file.lower().endswith((".jpg",".jpeg",".png")):
                sources.add(get_source_id(file))
        split_sources[split]=sources
    print("\nDataset:",dataset)
    splits=list(split_sources.keys())
    for i in range(len(splits)):
        for j in range(i+1,len(splits)):
            s1=splits[i]
            s2=splits[j]
            overlap=split_sources[s1] & split_sources[s2]
            print(s1,"vs",s2,":",len(overlap),"shared source(s)")


Dataset: DIATAquarium.v4i.yolov8
train vs valid : 3 shared source(s)

Dataset: Aquatic Plant.v2i.yolov8
train vs valid : 2 shared source(s)
train vs test : 2 shared source(s)
valid vs test : 2 shared source(s)

Dataset: well.v8i.yolov8
train vs valid : 1 shared source(s)
train vs test : 1 shared source(s)
valid vs test : 1 shared source(s)


In [ ]:
for dataset in DATASETS:
    print("\nDataset:",dataset)
    split_sources={}
    for split in ["train","valid","test"]:
        image_dir=os.path.join(DATASET_FOLDER,dataset,split,"images")
        if not os.path.exists(image_dir):
            continue
        sources=set()
        for file in os.listdir(image_dir):
            if file.lower().endswith((".jpg",".jpeg",".png")):
                sources.add(get_source_id(file))
        split_sources[split]=sources
    splits=list(split_sources.keys())
    for i in range(len(splits)):
        for j in range(i+1,len(splits)):
            shared=split_sources[splits[i]] & split_sources[splits[j]]
            if shared:
                print(splits[i],"vs",splits[j],":")
                for source in shared:
                    print(" ",source)


Dataset: DIATAquarium.v4i.yolov8
train vs valid :
  av0_0_20231228_102111_mp4
  av0_0_20231228_103918_mp4
  av0_0_20231228_094444_mp4

Dataset: Aquatic Plant.v2i.yolov8
train vs valid :
  2023-12-09_13_18_14_mp4
  2023-12-09_13_20_33_mp4
train vs test :
  2023-12-09_13_18_14_mp4
  2023-12-09_13_20_33_mp4
valid vs test :
  2023-12-09_13_18_14_mp4
  2023-12-09_13_20_33_mp4

Dataset: well.v8i.yolov8
train vs valid :
  20240102-20240102153559-20240102153818-153600_mp4
train vs test :
  20240102-20240102153559-20240102153818-153600_mp4
valid vs test :
  20240102-20240102153559-20240102153818-153600_mp4


In [ ]:
for dataset in DATASETS:
    print("\nDataset:",dataset)
    for source in set():
        pass
    # Collect all source IDs
    split_sources={}
    for split in ["train","valid","test"]:
        image_dir=os.path.join(DATASET_FOLDER,dataset,split,"images")
        if not os.path.exists(image_dir):
            continue
        counts={}
        for file in os.listdir(image_dir):
            if file.lower().endswith((".jpg",".jpeg",".png")):
                source=get_source_id(file)
                counts[source]=counts.get(source,0)+1
        split_sources[split]=counts
    # Find sources shared across splits
    splits=list(split_sources.keys())
    for i in range(len(splits)):
        for j in range(i+1,len(splits)):
            s1,s2=splits[i],splits[j]
            shared=set(split_sources[s1]) & set(split_sources[s2])
            for source in shared:
                print(source)
                print(" ",s1,":",split_sources[s1][source],"images")
                print(" ",s2,":",split_sources[s2][source],"images")


Dataset: DIATAquarium.v4i.yolov8
av0_0_20231228_102111_mp4
  train : 1335 images
  valid : 130 images
av0_0_20231228_103918_mp4
  train : 501 images
  valid : 39 images
av0_0_20231228_094444_mp4
  train : 1566 images
  valid : 155 images

Dataset: Aquatic Plant.v2i.yolov8
2023-12-09_13_18_14_mp4
  train : 585 images
  valid : 61 images
2023-12-09_13_20_33_mp4
  train : 1294 images
  valid : 118 images
2023-12-09_13_18_14_mp4
  train : 585 images
  test : 26 images
2023-12-09_13_20_33_mp4
  train : 1294 images
  test : 63 images
2023-12-09_13_18_14_mp4
  valid : 61 images
  test : 26 images
2023-12-09_13_20_33_mp4
  valid : 118 images
  test : 63 images

Dataset: well.v8i.yolov8
20240102-20240102153559-20240102153818-153600_mp4
  train : 3780 images
  valid : 191 images
20240102-20240102153559-20240102153818-153600_mp4
  train : 3780 images
  test : 184 images
20240102-20240102153559-20240102153818-153600_mp4
  valid : 191 images
  test : 184 images


In [ ]:
for dataset in DATASETS:
    print("\nDataset:",dataset)
    for split in ["train","valid","test"]:
        image_dir=os.path.join(DATASET_FOLDER,dataset,split,"images")
        if not os.path.exists(image_dir):
            continue
        counts={}
        for file in os.listdir(image_dir):
            if file.lower().endswith((".jpg",".jpeg",".png")):
                source=get_source_id(file)
                counts[source]=counts.get(source,0)+1
        print("\n",split)
        for source,count in sorted(counts.items(),key=lambda x:-x[1]):
            if count>20:
                print(source,":",count)


Dataset: DIATAquarium.v4i.yolov8

 train
av0_0_20231228_094444_mp4 : 1566
av0_0_20231228_102111_mp4 : 1335
av0_0_20231228_103918_mp4 : 501

 valid
av0_0_20231228_094444_mp4 : 155
av0_0_20231228_102111_mp4 : 130
av0_0_20231228_103918_mp4 : 39

Dataset: Aquatic Plant.v2i.yolov8

 train
2023-12-09_13_20_33_mp4 : 1294
2023-12-09_13_18_14_mp4 : 585

 valid
2023-12-09_13_20_33_mp4 : 118
2023-12-09_13_18_14_mp4 : 61

 test
2023-12-09_13_20_33_mp4 : 63
2023-12-09_13_18_14_mp4 : 26

Dataset: well.v8i.yolov8

 train
20240102-20240102153559-20240102153818-153600_mp4 : 3780

 valid
20240102-20240102153559-20240102153818-153600_mp4 : 191

 test
20240102-20240102153559-20240102153818-153600_mp4 : 184


## Leakage-Safe Split Plan
The existing dataset splits contain frames from the same source across train, validation and test folders. This creates a potential source-level data leakage risk.
For Dataset V1, source groups will be kept within a single split wherever feasible. The original raw datasets will remain unchanged.

In [ ]:
SPLIT_PLAN=os.path.join(DATASET_V1,"split_plan")
os.makedirs(SPLIT_PLAN,exist_ok=True)
print("Split plan folder:",SPLIT_PLAN)

Split plan folder: /content/drive/MyDrive/Dataset_V1/split_plan


In [ ]:
for dataset in DATASETS:
    print("\nDataset:",dataset)
    source_splits={}
    for split in ["train","valid","test"]:
        image_dir=os.path.join(DATASET_FOLDER,dataset,split,"images")
        if not os.path.exists(image_dir):
            continue
        for file in os.listdir(image_dir):
            if file.lower().endswith((".jpg",".jpeg",".png")):
                source=get_source_id(file)
                if source not in source_splits:
                    source_splits[source]=set()
                source_splits[source].add(split)

    print("Sources appearing in multiple splits:")
    for source,splits in source_splits.items():
        if len(splits)>1:
            print(source,"->",sorted(splits))


Dataset: DIATAquarium.v4i.yolov8
Sources appearing in multiple splits:
av0_0_20231228_103918_mp4 -> ['train', 'valid']
av0_0_20231228_102111_mp4 -> ['train', 'valid']
av0_0_20231228_094444_mp4 -> ['train', 'valid']

Dataset: Aquatic Plant.v2i.yolov8
Sources appearing in multiple splits:
2023-12-09_13_20_33_mp4 -> ['test', 'train', 'valid']
2023-12-09_13_18_14_mp4 -> ['test', 'train', 'valid']

Dataset: well.v8i.yolov8
Sources appearing in multiple splits:
20240102-20240102153559-20240102153818-153600_mp4 -> ['test', 'train', 'valid']


##  Source-Level Class Coverage

Before creating Dataset V1 splits, the class distribution of the identified source groups is checked.

This helps determine whether source-based splitting can maintain sufficient class coverage in train, validation and test sets.

In [ ]:
import os
import re

def get_source_id(filename):
    name=os.path.splitext(filename)[0]
    name=re.sub(r"\s+\(\d+\)$","",name)
    name=re.sub(r"\.rf\.[a-f0-9]+$","",name)
    name=re.sub(r"_jpg$","",name)
    name=re.sub(r"-\d+$","",name)
    name=re.sub(r"_\d{6}$","",name)
    return name

for dataset in DATASETS:
    print("\nDataset:",dataset)
    dataset_path=os.path.join(DATASET_FOLDER,dataset)
    all_sources=set()

    for split in ["train","valid","test"]:
        image_dir=os.path.join(dataset_path,split,"images")

        if not os.path.exists(image_dir):
            continue

        for file in os.listdir(image_dir):
            if file.lower().endswith((".jpg",".jpeg",".png")):
                all_sources.add(get_source_id(file))

    print("Total source IDs:",len(all_sources))

    for source in sorted(all_sources):
        print(" ",source)


Dataset: DIATAquarium.v4i.yolov8
Total source IDs: 6
  av0_0_20231228_094444_mp4
  av0_0_20231228_102111_mp4
  av0_0_20231228_103918_mp4
  av0_0_20240222_102152
  av0_0_20240222_103703
  av0_0_20240222_105838

Dataset: Aquatic Plant.v2i.yolov8
Total source IDs: 2
  2023-12-09_13_18_14_mp4
  2023-12-09_13_20_33_mp4

Dataset: well.v8i.yolov8
Total source IDs: 1
  20240102-20240102153559-20240102153818-153600_mp4


In [ ]:
from collections import defaultdict
for dataset in DATASETS:
    print("\n______________________________")
    print("Dataset:",dataset)
    print("________________________________")
    dataset_path=os.path.join(DATASET_FOLDER,dataset)
    source_splits=defaultdict(set)
    for split in ["train","valid","test"]:
        image_dir=os.path.join(dataset_path,split,"images")
        if not os.path.exists(image_dir):
            continue
        for file in os.listdir(image_dir):
            if file.lower().endswith((".jpg",".jpeg",".png")):
                source=get_source_id(file)
                source_splits[source].add(split)
    for source,splits in sorted(source_splits.items()):
        print(source,"->",", ".join(sorted(splits)))


______________________________
Dataset: DIATAquarium.v4i.yolov8
________________________________
av0_0_20231228_094444_mp4 -> train, valid
av0_0_20231228_102111_mp4 -> train, valid
av0_0_20231228_103918_mp4 -> train, valid
av0_0_20240222_102152 -> train, valid
av0_0_20240222_103703 -> train, valid
av0_0_20240222_105838 -> train, valid

______________________________
Dataset: Aquatic Plant.v2i.yolov8
________________________________
2023-12-09_13_18_14_mp4 -> test, train, valid
2023-12-09_13_20_33_mp4 -> test, train, valid

______________________________
Dataset: well.v8i.yolov8
________________________________
20240102-20240102153559-20240102153818-153600_mp4 -> test, train, valid


In [ ]:
from collections import defaultdict,Counter
import os
for dataset in DATASETS:
    print("\n______________________________")
    print("Dataset:",dataset)
    print("________________________________")
    dataset_path=os.path.join(DATASET_FOLDER,dataset)
    source_classes=defaultdict(Counter)
    for split in ["train","valid","test"]:
        image_dir=os.path.join(dataset_path,split,"images")
        label_dir=os.path.join(dataset_path,split,"labels")
        if not os.path.exists(image_dir):
            continue
        files=[f for f in os.listdir(image_dir)
               if f.lower().endswith((".jpg",".jpeg",".png"))]
        print(split,":",len(files),"images")
        for i,file in enumerate(files):
            source=get_source_id(file)
            label_file=os.path.join(
                label_dir,
                os.path.splitext(file)[0]+".txt"
            )
            if os.path.exists(label_file):
                with open(label_file,"r") as f:
                    for line in f:
                        values=line.strip().split()
                        if len(values)==5:
                            class_id=int(float(values[0]))
                            source_classes[source][class_id]+=1
            if (i+1)%1000==0:
                print("  Processed:",i+1)
    print("\nSource-wise class coverage:")
    for source,counts in sorted(source_classes.items()):
        print(source,"->",dict(counts))


______________________________
Dataset: DIATAquarium.v4i.yolov8
________________________________
train : 8121 images
  Processed: 1000
  Processed: 2000
  Processed: 3000
  Processed: 4000
  Processed: 5000
  Processed: 6000
  Processed: 7000
  Processed: 8000
valid : 773 images

Source-wise class coverage:
av0_0_20231228_094444_mp4 -> {0: 2004, 3: 1474, 7: 196, 4: 275, 8: 485}
av0_0_20231228_102111_mp4 -> {2: 659, 8: 669, 0: 970, 9: 96, 1: 111, 3: 138}
av0_0_20231228_103918_mp4 -> {2: 108, 0: 237, 8: 72, 7: 14, 3: 210, 4: 16, 1: 15}
av0_0_20240222_102152 -> {5: 115, 0: 1182, 8: 641, 6: 850, 2: 366, 10: 607, 7: 225, 3: 40, 9: 95, 1: 3, 4: 21}
av0_0_20240222_103703 -> {10: 958, 0: 12, 3: 3, 6: 1585, 2: 674, 5: 14}
av0_0_20240222_105838 -> {0: 333, 2: 171, 5: 1019, 6: 233, 3: 588, 4: 39, 7: 25, 8: 19, 9: 1}

______________________________
Dataset: Aquatic Plant.v2i.yolov8
________________________________
train : 1892 images
  Processed: 1000
valid : 179 images
test : 89 images

Source-w

## Final Split Strategy

The supplied datasets already contain Train, Validation and Test folders, but the source-level audit found that the same source appears across multiple splits.

DIATAquarium:
The dataset contains 6 source IDs. All 6 appear in both Train and Validation. A separate local Test split is not available.

Aquatic Plant:
The dataset contains 2 source IDs, and both appear across Train, Validation and Test. Therefore, three completely source-independent splits are not possible.

Well:
The dataset contains only 1 source ID, and it appears across Train, Validation and Test. Therefore, source-independent splitting is not possible.

For Dataset V1, the original raw dataset will be preserved. Existing splits will not be changed until the final split handling is confirmed. The limitation and source-level leakage will be documented and considered during model evaluation.

In [ ]:
import os
import shutil
DATASET_V1="/content/drive/MyDrive/Dataset_V1"
for dataset in DATASETS:
    source=os.path.join(DATASET_FOLDER,dataset)
    destination=os.path.join(DATASET_V1,dataset)
    if os.path.exists(destination):
        print(dataset,"-> already exists")
    else:
        shutil.copytree(source,destination)
        print(dataset,"-> copied to Dataset_V1")
print("\nDataset V1 preparation copy completed.")

DIATAquarium.v4i.yolov8 -> copied to Dataset_V1
Aquatic Plant.v2i.yolov8 -> copied to Dataset_V1
well.v8i.yolov8 -> copied to Dataset_V1

Dataset V1 preparation copy completed.


In [ ]:
for dataset in DATASETS:
    print("\n____________________________________")
    print("Dataset:",dataset)
    print("____________________________________")

    dataset_path=os.path.join(DATASET_V1,dataset)

    for split in ["train","valid","test"]:
        image_dir=os.path.join(dataset_path,split,"images")
        label_dir=os.path.join(dataset_path,split,"labels")

        if not os.path.exists(image_dir):
            print(split,": NOT PRESENT")
            continue

        image_count=len([
            f for f in os.listdir(image_dir)
            if f.lower().endswith((".jpg",".jpeg",".png"))
        ])

        label_count=len([
            f for f in os.listdir(label_dir)
            if f.lower().endswith(".txt")
        ]) if os.path.exists(label_dir) else 0

        print(split,":",image_count,"images,",label_count,"labels")


____________________________________
Dataset: DIATAquarium.v4i.yolov8
____________________________________
train : 8121 images, 8121 labels
valid : 773 images, 773 labels
test : NOT PRESENT

____________________________________
Dataset: Aquatic Plant.v2i.yolov8
____________________________________
train : 1892 images, 1879 labels
valid : 179 images, 179 labels
test : 89 images, 89 labels

____________________________________
Dataset: well.v8i.yolov8
____________________________________
train : 3795 images, 3780 labels
valid : 191 images, 191 labels
test : 184 images, 184 labels


In [ ]:
import os
for dataset in DATASETS:
    print("\n____________________________________")
    print("Dataset:",dataset)
    print("____________________________________")
    dataset_path=os.path.join(DATASET_V1,dataset)
    total_labels=0
    empty_labels=0
    invalid_labels=0
    for split in ["train","valid","test"]:
        label_dir=os.path.join(dataset_path,split,"labels")
        if not os.path.exists(label_dir):
            continue
        for file in os.listdir(label_dir):
            if not file.endswith(".txt"):
                continue
            total_labels+=1
            with open(os.path.join(label_dir,file),"r") as f:
                lines=f.readlines()
            if len(lines)==0:
                empty_labels+=1
                continue
            for line in lines:
                values=line.strip().split()
                if len(values)!=5:
                    invalid_labels+=1
                    continue
                try:
                    class_id=float(values[0])
                    x=float(values[1])
                    y=float(values[2])
                    w=float(values[3])
                    h=float(values[4])
                    if class_id<0 or x<0 or x>1 or y<0 or y>1 or w<=0 or w>1 or h<=0 or h>1:
                        invalid_labels+=1
                except:
                    invalid_labels+=1
    print("Total label files:",total_labels)
    print("Empty label files:",empty_labels)
    print("Invalid label entries:",invalid_labels)


____________________________________
Dataset: DIATAquarium.v4i.yolov8
____________________________________
Total label files: 8894
Empty label files: 657
Invalid label entries: 0

____________________________________
Dataset: Aquatic Plant.v2i.yolov8
____________________________________
Total label files: 2147
Empty label files: 1010
Invalid label entries: 0

____________________________________
Dataset: well.v8i.yolov8
____________________________________
Total label files: 4155
Empty label files: 39
Invalid label entries: 0


In [ ]:
import os

config_path=os.path.join(DATASET_V1,"preprocessing_config.txt")

with open(config_path,"w") as f:
    f.write("Dataset V1 Preprocessing Configuration\n")
    f.write("__________________________________________\n")
    f.write("Image size: 640 x 640\n")
    f.write("Pixel normalization: 0-1\n")
    f.write("Color format: RGB\n")
    f.write("Raw dataset modified: No\n")
    f.write("Datasets: DIATAquarium, Aquatic Plant, Well\n")
    f.write("Source-level leakage: Identified and documented\n")
    f.write("Augmentation: To be applied during model training\n")

print("Configuration saved at:")
print(config_path)

Configuration saved at:
/content/drive/MyDrive/Dataset_V1/preprocessing_config.txt


##Data Augmentation

Data augmentation will be applied during model training rather than permanently modifying the raw images.

Planned augmentations:
- Horizontal flipping
- Small rotation
- Scale variation
- Brightness and contrast variation

Augmentation will be applied only to training data. Validation and test data will remain unchanged for fair evaluation.

In [ ]:
AUGMENTATION_CONFIG={
    "horizontal_flip":True,
    "small_rotation":True,
    "scale_variation":True,
    "brightness_contrast":True,
    "apply_to":"train_only"
}
print("Augmentation configuration:")
print(AUGMENTATION_CONFIG)

Augmentation configuration:
{'horizontal_flip': True, 'small_rotation': True, 'scale_variation': True, 'brightness_contrast': True, 'apply_to': 'train_only'}


In [ ]:
import os
print("____________________________________")
print("FINAL DATASET V1 VERIFICATION")
print("____________________________________")
for dataset in DATASETS:
    print("\nDataset:",dataset)
    dataset_path=os.path.join(DATASET_V1,dataset)
    for split in ["train","valid","test"]:
        image_dir=os.path.join(dataset_path,split,"images")
        if not os.path.exists(image_dir):
            print(split,": NOT PRESENT")
            continue
        image_count=len([
            f for f in os.listdir(image_dir)
            if f.lower().endswith((".jpg",".jpeg",".png"))
        ])
        print(split,":",image_count,"images")
print("\nPreprocessing configuration exists:",
      os.path.exists(os.path.join(DATASET_V1,"preprocessing_config.txt")))

print("Dataset V1 verification completed.")

____________________________________
FINAL DATASET V1 VERIFICATION
____________________________________

Dataset: DIATAquarium.v4i.yolov8
train : 8121 images
valid : 773 images
test : NOT PRESENT

Dataset: Aquatic Plant.v2i.yolov8
train : 1892 images
valid : 179 images
test : 89 images

Dataset: well.v8i.yolov8
train : 3795 images
valid : 191 images
test : 184 images

Preprocessing configuration exists: True
Dataset V1 verification completed.


# Day 11 — Dataset V1 Preparation Summary

Dataset V1 preparation was completed for all three supplied underwater datasets.

The preprocessing setup includes:
- Image resizing to 640 × 640
- Pixel normalization to the range 0–1
- RGB image conversion
- A reusable dataset loader for images and YOLO annotations
- Training-only data augmentation configuration
- Preservation of the original raw datasets

A source-level leakage audit was also performed. The analysis showed that the supplied splits contain overlapping source IDs across Train, Validation and/or Test. A completely source-independent split is not feasible for Aquatic Plant and Well because they contain only 2 and 1 source IDs respectively.

Therefore, the raw dataset and existing split structure are preserved, and the source-level limitation will be considered during model evaluation.

Dataset V1 verification was completed successfully.